# **DEG ANALYSIS**

In [ ]:
# Lock Kaggle's loaded core C-libraries, then install scanpy safely
!pip freeze | grep -E "numpy|scipy|pandas" > constraints.txt
!pip install scanpy -c constraints.txt -q

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import os

sc.settings.verbosity = 2

# Primary path
h5ad_path = "/kaggle/input/datasets/ericdu847/microglial-annotated-multiregion-nuclei-data/SEAAD_Immune_HIP_MEC_LEC_ITG_MTG_FI_STG_DFC_AnG_V1C_RNAseq_final-nuclei.2026-06-22.h5ad"

# Fallback path check if Kaggle mounts under standard slug directory
if not os.path.exists(h5ad_path):
    fallback_paths = [
        "/kaggle/input/microglial-annotated-multiregion-nuclei-data/SEAAD_Immune_HIP_MEC_LEC_ITG_MTG_FI_STG_DFC_AnG_V1C_RNAseq_final-nuclei.2026-06-22.h5ad",
        "/kaggle/input/Microglial annotated multiregion nuclei data/SEAAD_Immune_HIP_MEC_LEC_ITG_MTG_FI_STG_DFC_AnG_V1C_RNAseq_final-nuclei.2026-06-22.h5ad"
    ]
    for alt in fallback_paths:
        if os.path.exists(alt):
            h5ad_path = alt
            break

if not os.path.exists(h5ad_path):
    raise FileNotFoundError(f"Could not locate the dataset file. Verified path: {h5ad_path}")

print(f"Loading dataset from: {h5ad_path}")
adata = sc.read_h5ad(h5ad_path)
print("Data loaded successfully. Shape:", adata.shape)

In [ ]:
# 1. Identify Diagnosis column safely
diag_col = None
for col in adata.obs.columns:
    if adata.obs[col].dtype.name in ['category', 'object', 'string']:
        unique_vals = adata.obs[col].dropna().astype(str).unique()
        if 'Dementia' in unique_vals and 'No dementia' in unique_vals:
            diag_col = col
            break

# 2. Identify Brain Region column safely
region_col = None
region_keywords = ['region', 'roi', 'structure']
for col in adata.obs.columns:
    if any(keyword in col.lower() for keyword in region_keywords):
        region_col = col
        break

# Safe fallbacks that check existing column names
if diag_col is None or diag_col not in adata.obs.columns:
    diag_candidates = [c for c in adata.obs.columns if any(k in c.lower() for k in ['cognitive', 'diagnosis', 'dementia'])]
    diag_col = diag_candidates[0] if diag_candidates else adata.obs.columns[0]

if region_col is None or region_col not in adata.obs.columns:
    region_candidates = [c for c in adata.obs.columns if any(k in c.lower() for k in ['region', 'structure', 'area'])]
    region_col = region_candidates[0] if region_candidates else adata.obs.columns[1]

print(f"Using Diagnosis Column: '{diag_col}'")
print(f"Using Region Column: '{region_col}'")

# Display distribution safely
print("\nNuclei count per Region & Diagnosis:")
print(pd.crosstab(adata.obs[region_col], adata.obs[diag_col]))

In [ ]:
# Ensure categorical target columns are strings
adata.obs[diag_col] = adata.obs[diag_col].astype(str)
adata.obs[region_col] = adata.obs[region_col].astype(str)

target_group = "Dementia"
ref_group = "No dementia"

regions = adata.obs[region_col].dropna().unique()
output_files = []

for region in regions:
    print(f"\n{'-'*40}")
    print(f"Processing Region: {region}")
    
    # --- 1. Subset and count ---
    # Isolate data for just the current region and the two diagnosis groups
    subset = adata[(adata.obs[region_col] == region) & 
                   (adata.obs[diag_col].isin([target_group, ref_group]))].copy()
    
    # Verify we have enough cells to run a valid statistical test
    counts = subset.obs[diag_col].value_counts()
    if target_group not in counts or ref_group not in counts:
        print(f"Skipping {region}: Missing target or reference group.")
        continue
    
    min_cells = 10
    if counts[target_group] < min_cells or counts[ref_group] < min_cells:
        print(f"Skipping {region}: Insufficient nuclei counts ({target_group}: {counts.get(target_group, 0)}, {ref_group}: {counts.get(ref_group, 0)})")
        continue

    print(f"Nuclei count -> {target_group}: {counts[target_group]} | {ref_group}: {counts[ref_group]}")
    
    # --- 2. Extract raw data for accurate calculations ---
    # Check if the unscaled, original counts are stored in .raw and pull them out
    if subset.raw is not None:
        deg_adata = sc.AnnData(X=subset.raw.X, obs=subset.obs, var=subset.raw.var)
    else:
        deg_adata = subset.copy()
        
    # --- 3. Normalize and log-transform if data is raw ---
    # A max value > 50 is a standard heuristic to detect unlogged raw counts
    if deg_adata.X.max() > 50:
        sc.pp.normalize_total(deg_adata, target_sum=1e4)
        sc.pp.log1p(deg_adata)
    
    # --- 4. Run t-test on the properly log-normalized data ---
    # The t-test ensures p-values and log fold changes are positively correlated
    sc.tl.rank_genes_groups(
        deg_adata,
        groupby=diag_col,
        reference=ref_group,
        method='t-test',
        pts=True,
        use_raw=False  # Explicitly False because we prepared deg_adata.X ourselves
    )
    
    # --- 5. Extract and format results for IPA ---
    deg_df = sc.get.rank_genes_groups_df(deg_adata, group=target_group)
    
    # Build the final table including the FDR (pvals_adj)
    ipa_df = pd.DataFrame({
        'gene_id': deg_df['names'],
        'logfoldchanges': deg_df['logfoldchanges'],
        'pvals': deg_df['pvals'],
        'pvals_adj': deg_df['pvals_adj']
    })
    
    # Clean IPA-breaking values (zeros crash log math in IPA)
    ipa_df['logfoldchanges'] = ipa_df['logfoldchanges'].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    ipa_df['pvals'] = ipa_df['pvals'].fillna(1.0).replace(0.0, 1e-300)
    ipa_df['pvals_adj'] = ipa_df['pvals_adj'].fillna(1.0).replace(0.0, 1e-300)
    
    # --- 6. Save CSV ---
    safe_region_name = region.replace(" ", "_").replace("/", "_")
    filename = f"{safe_region_name}_Dementia_vs_Nodementia_IPA.csv"
    ipa_df.to_csv(filename, index=False)
    output_files.append(filename)
    
    print(f"Saved: {filename}")
    
print(f"\n{'-'*40}")
print(f"Pipeline complete! Generated {len(output_files)} region-specific CSV files for IPA.")